# Lab 10. State-Space Models and Kalman Filtering

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-10-state-space-kalman-filtering-lab.ipynb)

This lab is designed for **independent study**. It includes background before programming, guided code, interpretation checkpoints, and short exercises. You should be able to read the explanations, run the code, and modify the examples on your own.

## Learning goals

By the end of this lab, you should be able to:

1. explain the difference between an observed series and a hidden state,
2. write a simple state-space model using a state equation and an observation equation,
3. implement the Kalman filter from scratch for a local-level model,
4. interpret prediction, innovation, update, and filtering uncertainty,
5. handle missing observations naturally using the Kalman filter,
6. estimate noise parameters by a likelihood grid search,
7. implement a local-linear trend model with a two-dimensional state,
8. connect state-space models to forecasting, smoothing, ARIMA, and sequence learning.

Throughout the lab, use the notation $X_t$ for a hidden state and $Y_t$ for an observed value.

## 1. Background: hidden states and noisy observations

Many time series are noisy measurements of a simpler hidden process. For example:

- a thermometer observes temperature with measurement error,
- a sensor observes the position of an object with random noise,
- a financial price may be viewed as a noisy observation of a latent trend,
- an economic indicator may be a noisy observation of an unobserved business-cycle state.

A **state-space model** separates two ideas:

1. the hidden state evolves over time,
2. the observed data are noisy measurements of the hidden state.

The general linear Gaussian state-space model is

$$
X_t = F X_{t-1} + W_t, \qquad W_t \sim N(0,Q),
$$

$$
Y_t = H X_t + V_t, \qquad V_t \sim N(0,R).
$$

Here:

- $X_t$ is the hidden state,
- $Y_t$ is the observed time series,
- $F$ describes how the hidden state moves forward,
- $H$ describes how the hidden state is observed,
- $Q$ is the state-noise variance or covariance,
- $R$ is the observation-noise variance or covariance.

The Kalman filter recursively computes the best Gaussian estimate of the hidden state after seeing observations up to time $t$.

## 2. The local-level model

We start with the scalar **local-level model**:

$$
X_t = X_{t-1} + W_t, \qquad W_t \sim N(0,q),
$$

$$
Y_t = X_t + V_t, \qquad V_t \sim N(0,r).
$$

The hidden state $X_t$ is a random walk. The observation $Y_t$ is the hidden state plus measurement noise.

The two parameters have clear meanings:

- $q$ controls how much the hidden level is allowed to move over time,
- $r$ controls how noisy the observed measurements are.

When $q$ is small and $r$ is large, the filter produces a smooth estimate. When $q$ is large and $r$ is small, the filter follows the observations more closely.

## 3. Setup

Run this cell first. It imports packages and defines a fixed random generator so that results are reproducible.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

## 4. Simulate a local-level model

We first simulate the hidden state $X_t$ and the observed series $Y_t$. In real data, we only observe $Y_t$. In simulation, we know $X_t$, so we can check whether the filter is recovering the hidden state.

In [ ]:
def simulate_local_level(n=200, q=0.04, r=1.0, x0=0.0, seed=7339):
    local_rng = np.random.default_rng(seed)
    x = np.zeros(n)
    y = np.zeros(n)
    x[0] = x0 + local_rng.normal(0, np.sqrt(q))
    y[0] = x[0] + local_rng.normal(0, np.sqrt(r))

    for t in range(1, n):
        x[t] = x[t - 1] + local_rng.normal(0, np.sqrt(q))
        y[t] = x[t] + local_rng.normal(0, np.sqrt(r))

    return x, y

n = 220
q_true = 0.04
r_true = 1.00
x_true, y_obs = simulate_local_level(n=n, q=q_true, r=r_true)

time = np.arange(n)

plt.figure()
plt.plot(time, x_true, label="hidden state X_t")
plt.scatter(time, y_obs, s=12, alpha=0.55, label="observed Y_t")
plt.title("Simulated local-level model")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

pd.DataFrame({"hidden_state": x_true, "observed": y_obs}).head()

### Interpretation checkpoint

Look at the plot above.

1. Which curve is smoother: the hidden state or the observations?
2. Why is the observed series more variable?
3. If you only saw the observations, would you be able to recover the hidden state exactly?

The answer to the third question is no. The goal is not perfect recovery. The goal is a principled estimate with uncertainty.

## 5. Kalman filter equations for the local-level model

Suppose that after time $t-1$ we have

$$
X_{t-1} \mid Y_1,\ldots,Y_{t-1} \sim N(m_{t-1}, P_{t-1}).
$$

The Kalman filter has two steps.

### Prediction step

Before observing $Y_t$:

$$
a_t = m_{t-1}, \qquad P_t^- = P_{t-1} + q.
$$

Here $a_t$ is the predicted state mean and $P_t^-$ is the predicted state variance.

### Update step

After observing $Y_t$:

$$
e_t = Y_t - a_t,
$$

$$
S_t = P_t^- + r,
$$

$$
K_t = \frac{P_t^-}{S_t},
$$

$$
m_t = a_t + K_t e_t,
$$

$$
P_t = (1-K_t)P_t^-.
$$

The quantity $e_t$ is called the **innovation** or one-step-ahead forecast error. The quantity $K_t$ is the **Kalman gain**. It controls how much the filter trusts the new observation.

## 6. Implement the scalar Kalman filter from scratch

The function below returns a table containing predicted means, filtered means, variances, innovations, and Kalman gains.

In [ ]:
def kalman_filter_local_level(y, q, r, m0=0.0, P0=100.0):
    """Kalman filter for the scalar local-level model.

    Model:
        X_t = X_{t-1} + W_t, W_t ~ N(0, q)
        Y_t = X_t + V_t,     V_t ~ N(0, r)

    Missing observations are allowed and should be represented by np.nan.
    """
    y = np.asarray(y, dtype=float)
    n = len(y)

    pred_mean = np.zeros(n)
    pred_var = np.zeros(n)
    filt_mean = np.zeros(n)
    filt_var = np.zeros(n)
    innovation = np.full(n, np.nan)
    innovation_var = np.full(n, np.nan)
    gain = np.full(n, np.nan)
    loglik_terms = np.zeros(n)

    m_prev = float(m0)
    P_prev = float(P0)

    for t in range(n):
        # Prediction step
        a = m_prev
        P_pred = P_prev + q

        pred_mean[t] = a
        pred_var[t] = P_pred

        if np.isnan(y[t]):
            # No update if observation is missing
            m = a
            P = P_pred
            loglik_terms[t] = 0.0
        else:
            # Update step
            e = y[t] - a
            S = P_pred + r
            K = P_pred / S
            m = a + K * e
            P = (1.0 - K) * P_pred

            innovation[t] = e
            innovation_var[t] = S
            gain[t] = K
            loglik_terms[t] = -0.5 * (np.log(2 * np.pi) + np.log(S) + e**2 / S)

        filt_mean[t] = m
        filt_var[t] = P
        m_prev = m
        P_prev = P

    return pd.DataFrame({
        "y": y,
        "pred_mean": pred_mean,
        "pred_var": pred_var,
        "filt_mean": filt_mean,
        "filt_var": filt_var,
        "innovation": innovation,
        "innovation_var": innovation_var,
        "kalman_gain": gain,
        "loglik_term": loglik_terms
    })

kf = kalman_filter_local_level(y_obs, q=q_true, r=r_true)
kf.head(10)

## 7. Plot the filtered estimate and uncertainty bands

The filtered estimate is the estimate of $X_t$ after seeing data through time $t$.

The interval shown below is

$$
\hat X_t \pm 2 \sqrt{P_t}.
$$

It is not a classical confidence interval for a fixed parameter. It is a state uncertainty band: it describes uncertainty about the hidden state at time $t$.

In [ ]:
def plot_filter_result(time, y, x_true, kf_table, title="Kalman filter result"):
    m = kf_table["filt_mean"].to_numpy()
    P = kf_table["filt_var"].to_numpy()
    lower = m - 2 * np.sqrt(P)
    upper = m + 2 * np.sqrt(P)

    plt.figure(figsize=(11, 5))
    if x_true is not None:
        plt.plot(time, x_true, label="hidden state X_t")
    plt.scatter(time, y, s=12, alpha=0.45, label="observed Y_t")
    plt.plot(time, m, linewidth=2.2, label="filtered estimate")
    plt.fill_between(time, lower, upper, alpha=0.18, label="approximately ±2 std. errors")
    plt.title(title)
    plt.xlabel("time")
    plt.ylabel("value")
    plt.legend()
    plt.show()

plot_filter_result(time, y_obs, x_true, kf, title="Local-level Kalman filter")

rmse_observed = np.sqrt(np.mean((y_obs - x_true)**2))
rmse_filtered = np.sqrt(np.mean((kf["filt_mean"].to_numpy() - x_true)**2))
print(f"RMSE of observations versus hidden state: {rmse_observed:.3f}")
print(f"RMSE of filtered estimate versus hidden state: {rmse_filtered:.3f}")

### Interpretation checkpoint

The filtered estimate should usually be closer to the hidden state than the noisy observations are. But this depends on the assumed values of $q$ and $r$.

The filter is balancing two sources of information:

- the previous filtered state estimate,
- the new observation.

The Kalman gain tells us how much the new observation changes the estimate.

## 8. Understand the Kalman gain

The Kalman gain is

$$
K_t = \frac{P_t^-}{P_t^- + r}.
$$

It is always between 0 and 1 in this scalar model.

- If $K_t$ is close to 1, the filter trusts the new observation strongly.
- If $K_t$ is close to 0, the filter trusts the previous prediction more strongly.

The gain changes during the early observations and often stabilizes later.

In [ ]:
plt.figure()
plt.plot(time, kf["kalman_gain"])
plt.title("Kalman gain over time")
plt.xlabel("time")
plt.ylabel("Kalman gain")
plt.ylim(0, 1.05)
plt.show()

print("First five gains:")
print(kf["kalman_gain"].head().round(3).to_string(index=False))
print("\nLast five gains:")
print(kf["kalman_gain"].tail().round(3).to_string(index=False))

## 9. Experiment: what happens when $q$ and $r$ are misspecified?

The same observations can lead to different filtered estimates if we change the assumed noise levels.

Try three filters:

1. small $q$, large $r$: very smooth estimate,
2. true $q$, true $r$: approximately correct balance,
3. large $q$, small $r$: estimate follows observations closely.

In [ ]:
settings = [
    (0.005, 2.0, "small q, large r"),
    (q_true, r_true, "true q, true r"),
    (0.50, 0.20, "large q, small r"),
]

plt.figure(figsize=(11, 5))
plt.plot(time, x_true, linewidth=2.0, label="hidden state")
plt.scatter(time, y_obs, s=10, alpha=0.25, label="observed")

for q_assumed, r_assumed, label in settings:
    temp = kalman_filter_local_level(y_obs, q=q_assumed, r=r_assumed)
    plt.plot(time, temp["filt_mean"], label=label)

plt.title("Effect of assumed state noise q and observation noise r")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Exercise 1

Change the three parameter settings above.

1. What happens when $q=0.001$ and $r=4$?
2. What happens when $q=2$ and $r=0.05$?
3. Which setting gives the best visual estimate of the hidden state?
4. Why is visual quality not enough for real data, where the hidden state is unknown?

## 10. Missing observations

One advantage of state-space models is that missing observations are easy to handle.

If $Y_t$ is missing, we skip the update step and keep the prediction:

$$
m_t = a_t, \qquad P_t = P_t^-.
$$

The uncertainty should increase during a missing block because the state keeps evolving but no new data arrive.

In [ ]:
y_missing = y_obs.copy()
y_missing[70:95] = np.nan

y_missing[145:155] = np.nan

kf_missing = kalman_filter_local_level(y_missing, q=q_true, r=r_true)

plt.figure(figsize=(11, 5))
plt.plot(time, x_true, linewidth=2.0, label="hidden state")
plt.scatter(time, y_missing, s=13, alpha=0.55, label="observed with missing values")
plt.plot(time, kf_missing["filt_mean"], linewidth=2.2, label="filtered estimate")

m = kf_missing["filt_mean"].to_numpy()
P = kf_missing["filt_var"].to_numpy()
plt.fill_between(time, m - 2*np.sqrt(P), m + 2*np.sqrt(P), alpha=0.18, label="approximately ±2 std. errors")
plt.title("Kalman filtering with missing observations")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

print("Filtered variance before, during, and after the first missing block:")
for idx in [68, 75, 90, 98]:
    print(f"t={idx:3d}, filtered variance={kf_missing.loc[idx, 'filt_var']:.3f}")

### Interpretation checkpoint

During the missing block, the filtered mean is just the model prediction. The filtered variance grows because the state equation keeps adding uncertainty.

This is one reason state-space models are useful in practice: they do not require filling missing observations before modeling.

## 11. Likelihood for parameter selection

For a fixed pair $(q,r)$, the Kalman filter produces innovations $e_t$ and innovation variances $S_t$. Under the Gaussian model,

$$
e_t \sim N(0,S_t).
$$

So the log likelihood contribution is

$$
-\frac{1}{2}\left[\log(2\pi) + \log(S_t) + \frac{e_t^2}{S_t}\right].
$$

We can add these terms over time and search for the values of $q$ and $r$ that fit the data best.

In [ ]:
def kalman_loglik_local_level(y, q, r, m0=0.0, P0=100.0):
    table = kalman_filter_local_level(y, q=q, r=r, m0=m0, P0=P0)
    return table["loglik_term"].sum()

q_grid = np.linspace(0.01, 0.20, 30)
r_grid = np.linspace(0.30, 2.50, 30)
loglik_grid = np.zeros((len(q_grid), len(r_grid)))

for i, q_value in enumerate(q_grid):
    for j, r_value in enumerate(r_grid):
        loglik_grid[i, j] = kalman_loglik_local_level(y_obs, q=q_value, r=r_value)

best_index = np.unravel_index(np.argmax(loglik_grid), loglik_grid.shape)
best_q = q_grid[best_index[0]]
best_r = r_grid[best_index[1]]

print(f"True q: {q_true:.3f}, grid-likelihood estimate of q: {best_q:.3f}")
print(f"True r: {r_true:.3f}, grid-likelihood estimate of r: {best_r:.3f}")

plt.figure(figsize=(8, 6))
plt.contourf(r_grid, q_grid, loglik_grid, levels=25)
plt.scatter([r_true], [q_true], s=80, marker="x", label="true value")
plt.scatter([best_r], [best_q], s=80, marker="o", label="best grid value")
plt.xlabel("r: observation-noise variance")
plt.ylabel("q: state-noise variance")
plt.title("Kalman-filter log likelihood grid")
plt.colorbar(label="log likelihood")
plt.legend()
plt.show()

### Exercise 2

The likelihood estimate may not exactly equal the true value because we only have one finite simulated series.

Try the following:

1. Increase `n` to 1000 and rerun the simulation and likelihood grid.
2. Make the grid narrower around the true values.
3. Explain why estimates improve on average as the sample size increases.

## 12. One-step-ahead forecasting

The Kalman filter naturally produces one-step-ahead predictions. In the local-level model, before seeing $Y_t$, the forecast is

$$
\hat Y_{t|t-1} = a_t.
$$

The forecast error is the innovation

$$
e_t = Y_t - \hat Y_{t|t-1}.
$$

A good model should have innovations that behave approximately like white noise.

In [ ]:
forecast = kf["pred_mean"].to_numpy()
innov = kf["innovation"].to_numpy()
valid = ~np.isnan(innov)

plt.figure(figsize=(11, 4))
plt.plot(time, y_obs, label="observed")
plt.plot(time, forecast, label="one-step-ahead forecast")
plt.title("One-step-ahead forecasts from the Kalman filter")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

plt.figure(figsize=(11, 4))
plt.axhline(0, linewidth=1)
plt.plot(time[valid], innov[valid])
plt.title("Innovations: observed minus one-step-ahead forecast")
plt.xlabel("time")
plt.ylabel("innovation")
plt.show()

print(f"Innovation mean: {np.nanmean(innov):.3f}")
print(f"Innovation standard deviation: {np.nanstd(innov):.3f}")

## 13. Local-linear trend model

The local-level model allows the level to move randomly, but it does not explicitly model a slope. A richer model uses a two-dimensional hidden state:

$$
X_t =
\begin{bmatrix}
\text{level}_t \\
\text{slope}_t
\end{bmatrix}.
$$

The state equation is

$$
X_t =
\begin{bmatrix}
1 & 1 \\
0 & 1
\end{bmatrix}
X_{t-1} + W_t.
$$

The observation equation is

$$
Y_t =
\begin{bmatrix}
1 & 0
\end{bmatrix}
X_t + V_t.
$$

This model is useful when the hidden series has a changing trend.

In [ ]:
def simulate_local_linear_trend(n=220, q_level=0.02, q_slope=0.002, r=0.50, seed=1234):
    local_rng = np.random.default_rng(seed)
    level = np.zeros(n)
    slope = np.zeros(n)
    y = np.zeros(n)

    level[0] = 0.0
    slope[0] = 0.04
    y[0] = level[0] + local_rng.normal(0, np.sqrt(r))

    for t in range(1, n):
        level[t] = level[t-1] + slope[t-1] + local_rng.normal(0, np.sqrt(q_level))
        slope[t] = slope[t-1] + local_rng.normal(0, np.sqrt(q_slope))
        y[t] = level[t] + local_rng.normal(0, np.sqrt(r))

    return level, slope, y

level_true, slope_true, y_trend = simulate_local_linear_trend()

plt.figure(figsize=(11, 5))
plt.plot(time, level_true, linewidth=2.2, label="hidden level")
plt.scatter(time, y_trend, s=12, alpha=0.45, label="observed")
plt.title("Simulated local-linear trend model")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

## 14. Implement a general linear Kalman filter

For the local-linear trend model, the state is a vector, so we use matrix equations.

Prediction:

$$
a_t = F m_{t-1}, \qquad P_t^- = F P_{t-1} F^T + Q.
$$

Update:

$$
e_t = Y_t - H a_t,
$$

$$
S_t = H P_t^- H^T + R,
$$

$$
K_t = P_t^- H^T S_t^{-1},
$$

$$
m_t = a_t + K_t e_t,
$$

$$
P_t = (I - K_t H)P_t^-.
$$

In [ ]:
def kalman_filter_linear_gaussian(y, F, H, Q, R, m0, P0):
    """General Kalman filter for a linear Gaussian state-space model.

    y is a one-dimensional observed series.
    F is the state transition matrix.
    H is the observation matrix with shape (1, state_dim).
    Q is the state-noise covariance matrix.
    R is a scalar observation-noise variance or a 1x1 array.
    """
    y = np.asarray(y, dtype=float)
    F = np.asarray(F, dtype=float)
    H = np.asarray(H, dtype=float)
    Q = np.asarray(Q, dtype=float)
    R = np.asarray([[float(R)]])
    m_prev = np.asarray(m0, dtype=float).reshape(-1, 1)
    P_prev = np.asarray(P0, dtype=float)

    n = len(y)
    state_dim = F.shape[0]

    pred_means = np.zeros((n, state_dim))
    filt_means = np.zeros((n, state_dim))
    filt_covs = np.zeros((n, state_dim, state_dim))
    innovations = np.full(n, np.nan)
    innovation_vars = np.full(n, np.nan)

    I = np.eye(state_dim)

    for t in range(n):
        a = F @ m_prev
        P_pred = F @ P_prev @ F.T + Q

        pred_means[t] = a.ravel()

        if np.isnan(y[t]):
            m = a
            P = P_pred
        else:
            e = np.array([[y[t]]]) - H @ a
            S = H @ P_pred @ H.T + R
            K = P_pred @ H.T @ np.linalg.inv(S)
            m = a + K @ e
            P = (I - K @ H) @ P_pred
            innovations[t] = e.item()
            innovation_vars[t] = S.item()

        filt_means[t] = m.ravel()
        filt_covs[t] = P
        m_prev = m
        P_prev = P

    return {
        "pred_means": pred_means,
        "filt_means": filt_means,
        "filt_covs": filt_covs,
        "innovations": innovations,
        "innovation_vars": innovation_vars
    }

F = np.array([[1.0, 1.0],
              [0.0, 1.0]])
H = np.array([[1.0, 0.0]])
Q = np.diag([0.02, 0.002])
R = 0.50
m0 = np.array([0.0, 0.0])
P0 = np.diag([100.0, 10.0])

kf_trend = kalman_filter_linear_gaussian(y_trend, F, H, Q, R, m0, P0)
filt_level = kf_trend["filt_means"][:, 0]
filt_slope = kf_trend["filt_means"][:, 1]

plt.figure(figsize=(11, 5))
plt.plot(time, level_true, linewidth=2.2, label="hidden level")
plt.scatter(time, y_trend, s=12, alpha=0.40, label="observed")
plt.plot(time, filt_level, linewidth=2.2, label="filtered level")
plt.title("Local-linear trend Kalman filter: level")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

plt.figure(figsize=(11, 4))
plt.plot(time, slope_true, linewidth=2.2, label="hidden slope")
plt.plot(time, filt_slope, linewidth=2.2, label="filtered slope")
plt.title("Local-linear trend Kalman filter: slope")
plt.xlabel("time")
plt.ylabel("slope")
plt.legend()
plt.show()

### Interpretation checkpoint

The local-linear trend model estimates both the level and the slope.

1. When does the slope increase or decrease?
2. Is the filtered slope noisier or smoother than the observed series?
3. Why might a two-dimensional state be better than a local-level model for trending data?

## 15. Compare with a simple rolling mean

A rolling mean is a useful smoother, but it is not a full probabilistic model. It does not produce a hidden-state equation, a measurement equation, or likelihood-based parameter estimates.

The comparison below shows the difference between a rolling average and a Kalman-filter estimate.

In [ ]:
rolling_mean = pd.Series(y_trend).rolling(window=15, center=True, min_periods=1).mean().to_numpy()

plt.figure(figsize=(11, 5))
plt.scatter(time, y_trend, s=12, alpha=0.35, label="observed")
plt.plot(time, level_true, linewidth=2.0, label="hidden level")
plt.plot(time, rolling_mean, linewidth=2.0, label="centered rolling mean")
plt.plot(time, filt_level, linewidth=2.2, label="Kalman filtered level")
plt.title("Rolling mean versus Kalman filtered level")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

rmse_roll = np.sqrt(np.mean((rolling_mean - level_true)**2))
rmse_kalman = np.sqrt(np.mean((filt_level - level_true)**2))
print(f"Rolling mean RMSE versus hidden level: {rmse_roll:.3f}")
print(f"Kalman filter RMSE versus hidden level: {rmse_kalman:.3f}")

## 16. AI-assisted study prompts

Use an AI assistant as a tutor, but verify the mathematics and code yourself.

### Prompt 1: concept explanation

> Explain the local-level state-space model in simple terms. What are $q$ and $r$? Why does the Kalman gain increase when observations are more reliable?

### Prompt 2: code review

> Review my scalar Kalman filter implementation. Check whether the prediction and update equations are correct. Also check whether missing observations are handled properly.

### Prompt 3: diagnostic reasoning

> I fit a local-level model and the innovations still show autocorrelation. Give three possible reasons and suggest model improvements.

### Prompt 4: model choice

> Compare a local-level model, a local-linear trend model, and ARIMA for a trending time series. What assumptions does each model make?

When using AI, do not simply accept the answer. Ask: Which equation supports this statement? Which variable is observable? Which variable is hidden? Does the code use future data accidentally?

## 17. Mini-project

Choose one of the following.

### Option A: Missing sensor data

Simulate a local-level model with two missing blocks. Fit the Kalman filter and explain how the uncertainty changes inside each missing block.

### Option B: Changing trend

Simulate a local-linear trend model. Compare the local-level and local-linear trend filters. Which model recovers the hidden level better?

### Option C: Parameter sensitivity

For the same observed series, fit several local-level filters with different $q/r$ ratios. Explain how the ratio changes smoothness and forecast responsiveness.

Your mini-project should include:

1. a clear problem statement,
2. one simulation or one real data example,
3. at least two plots,
4. one quantitative comparison such as RMSE or log likelihood,
5. a short interpretation paragraph.

## 18. Checklist

Before leaving this lab, make sure you can do the following without looking at the solution:

- [ ] Explain the difference between hidden states and observations.
- [ ] Write the local-level model equations.
- [ ] Identify the role of $q$ and $r$.
- [ ] Implement the scalar Kalman filter prediction step.
- [ ] Implement the scalar Kalman filter update step.
- [ ] Interpret the innovation and Kalman gain.
- [ ] Explain why missing observations are easy in state-space models.
- [ ] Use innovations to compute a log likelihood.
- [ ] Implement the matrix Kalman filter for a two-dimensional state.
- [ ] Explain when a local-linear trend model is better than a local-level model.